# Task 2: TF-IDF and Cosine Similarity
We turn each message into numbers with TF-IDF.
Then we use cosine similarity to find messages that are alike.
We use the clean text (clean_text), so we keep all the words.

## 1. Load the data

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
train = pd.read_csv('../data/train_prep_v2.csv')
test = pd.read_csv('../data/test_prep_v2.csv')
print(train.shape, test.shape)

(4102, 9) (1026, 9)


## 2. TF-IDF
TF-IDF gives each word in a message a number (a weight).
- TF: how many times the word is in this message.
- IDF: how rare the word is in all messages.
A word gets a big weight if it is in this message but rare in other messages.
A word gets a small weight if it is in almost every message, like "to".
We learn the words and the IDF from the train data only.
Then we use them on the test data.

In [2]:
vec = TfidfVectorizer(token_pattern=r'\S+')
X_train = vec.fit_transform(train['clean_text'])
X_test = vec.transform(test['clean_text'])
print('Train:', X_train.shape)
print('Test:', X_test.shape)

Train: (4102, 7303)
Test: (1026, 7303)


## 3. Why token_pattern=r'\S+'?
By default, TF-IDF in sklearn drops words with only one letter.
But one-letter words like "i", "u", "2" and "ü" are very different in ham and spam.
We check this.

In [3]:
default_vec = TfidfVectorizer()
default_vec.fit(train['clean_text'])
print('Words with default setting:', len(default_vec.vocabulary_))
print('Words with our setting:', len(vec.vocabulary_))
for w in ['i', 'u', '2', 'ü']:
    print(w, '| default:', w in default_vec.vocabulary_, '| ours:', w in vec.vocabulary_)

Words with default setting: 7266
Words with our setting: 7303
i | default: False | ours: True
u | default: False | ours: True
2 | default: False | ours: True
ü | default: False | ours: True


## 4. Common and rare words (IDF)
A small IDF means the word is in many messages.
A big IDF means the word is rare.

In [4]:
words = vec.get_feature_names_out()
idf = pd.Series(vec.idf_, index=words).sort_values()
print('Most common words:')
print(idf.head(8).round(2))
print('Rarest words:')
print(idf.tail(5).round(2))

Most common words:
i      2.16
to     2.20
you    2.26
a      2.58
the    2.68
and    2.92
u      2.93
in     2.95
dtype: float64
Rarest words:
messaging    8.63
cast         8.63
casting      8.63
mj           8.63
lf56         8.63
dtype: float64


## 5. TF-IDF of one message
We take one spam message and see which words got the biggest weight.

In [5]:
i = train[train['label'] == 'spam'].index[0]
print(train['clean_text'][i])
row = pd.Series(X_train[i].toarray()[0], index=words)
row = row[row > 0].sort_values(ascending=False)
print(row.head(8).round(3))

please call amanda with regard to renewing or upgrading your current t mobile handset free of charge offer ends today tel longnumber subject to tandc
amanda       0.284
renewing     0.284
upgrading    0.284
subject      0.284
regard       0.284
current      0.248
handset      0.243
ends         0.235
dtype: float64


## 6. Words in test that are not in train
TF-IDF only knows the words it learned from the train data.
If a word in a test message is new, TF-IDF ignores it.

In [6]:
test_words = ' '.join(test['clean_text']).split()
known = set(words)
new = [w for w in test_words if w not in known]
print('Words in test:', len(test_words))
print('Not in train:', len(new), '(', round(100 * len(new) / len(test_words), 1), '% )')

Words in test: 16109
Not in train: 1027 ( 6.4 % )


## 7. Cosine similarity
Cosine similarity tells how much two messages are alike.
It gives a number between 0 and 1:
- 1 means the two messages use the same words with the same weights.
- 0 means they share no word.
We compare every train message with every other train message.

In [7]:
sim = cosine_similarity(X_train)
np.fill_diagonal(sim, np.nan)
labels = train['label'].values
is_ham = labels == 'ham'
is_spam = labels == 'spam'
print('ham with ham  :', round(np.nanmean(sim[np.ix_(is_ham, is_ham)]), 3))
print('spam with spam:', round(np.nanmean(sim[np.ix_(is_spam, is_spam)]), 3))
print('ham with spam :', round(np.nanmean(sim[np.ix_(is_ham, is_spam)]), 3))

ham with ham  : 0.022
spam with spam: 0.064
ham with spam : 0.017


## 8. Find messages that are alike
We take a message from the test data.
Then we find the 3 train messages that are most like it.

In [8]:
first_spam = test[test['label'] == 'spam'].index[0]
first_ham = test[test['label'] == 'ham'].index[0]
for j in [first_spam, first_ham]:
    scores = cosine_similarity(X_test[j], X_train)[0]
    top = scores.argsort()[::-1][:3]
    print('TEST', test['label'][j], '|', test['text'][j][:90])
    for t in top:
        print('   ', round(scores[t], 3), train['label'][t], '|', train['text'][t][:80])
    print()

TEST spam | WINNER!! As a valued network customer you have been selected to receivea £900 prize reward
    0.642 spam | WINNER! As a valued network customer you hvae been selected to receive a £900 re
    0.382 spam | You are a winner you have been specially selected to receive £1000 cash or a £20
    0.382 spam | You are a winner you have been specially selected to receive £1000 cash or a £20

TEST ham | sir, you will receive the account no another 1hr time. Sorry for the delay.
    0.422 ham | Sorry for the delay. Yes masters
    0.289 ham | Goodmorning, today i am late for 1hr.
    0.238 ham | Sorry, I'll call you  later. I am in meeting sir.



## 9. Does the most similar message have the same label?
For every test message, we find the train message that is most like it.
Then we check if it has the same label.
This is a simple way to see if TF-IDF and cosine similarity are useful.

In [9]:
all_scores = cosine_similarity(X_test, X_train)
nearest = all_scores.argmax(axis=1)
same = train['label'].values[nearest] == test['label'].values
print('Same label:', round(100 * same.mean(), 1), '%')
print('For spam  :', round(100 * same[test['label'] == 'spam'].mean(), 1), '%')
print('For ham   :', round(100 * same[test['label'] == 'ham'].mean(), 1), '%')
best = pd.Series(all_scores.max(axis=1))
print(best.groupby(test['label'].values).mean().round(3))

Same label: 98.4 %
For spam  : 89.6 %
For ham   : 99.7 %
ham     0.390
spam    0.642
dtype: float64
